# 01 - Delta Fixture Walkthrough

Synthetic engineering and validation evidence, not final regulatory capital.

Use this notebook with the [SBM package journey](../docs/PACKAGE_JOURNEY.md) and the executable [package demo](../examples/run_demo.py).

Raw inputs your upstream risk engine must emit: canonical SBM sensitivity rows, or one Arrow table per homogeneous `(risk_class, risk_measure)` path, with deterministic sensitivity identifiers, desk and legal-entity lineage, bucket labels, risk-factor vertices, and signed sensitivity amounts. The machine-readable GIRR delta input-table example is [`frtb_sbm.girr_delta.schema.json`](../../../docs/schemas/input_table/frtb_sbm.girr_delta.schema.json); the package dataset contract is [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).

This notebook replays the `girr_delta_v1` fixture through the public row API, validates reconciliation, and inspects the deterministic audit payload. The fixture is synthetic and is not final regulatory capital output.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
PACKAGE_ROOT = None
REPO_ROOT = None
for candidate in (HERE, *HERE.parents):
    if (candidate / "src" / "frtb_sbm").exists():
        PACKAGE_ROOT = candidate
        REPO_ROOT = candidate.parent.parent if candidate.parent.name == "packages" else candidate
        break
    if (candidate / "packages" / "frtb-sbm" / "src" / "frtb_sbm").exists():
        REPO_ROOT = candidate
        PACKAGE_ROOT = candidate / "packages" / "frtb-sbm"
        break
if PACKAGE_ROOT is None:
    raise RuntimeError("Could not locate frtb-sbm package root")
workspace_src_paths = tuple(sorted((REPO_ROOT / "packages").glob("*/src"))) if REPO_ROOT else ()
for path in (
    PACKAGE_ROOT / "examples",
    PACKAGE_ROOT,
    PACKAGE_ROOT / "src",
    *workspace_src_paths,
    REPO_ROOT,
):
    if path is not None and str(path) not in sys.path:
        sys.path.insert(0, str(path))
PACKAGE_ROOT

In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:

    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)


from frtb_common import UnsupportedRegulatoryFeatureError
from frtb_sbm import (
    SbmInputError,
    calculate_sbm_capital,
    serialize_sbm_result,
    validate_sbm_result_reconciliation,
)
from sbm_notebook_data import load_fixture, markdown_table

## Public API happy path

This notebook replays a synthetic GIRR delta fixture through the top-level row API: `calculate_sbm_capital`.


In [ ]:
fixture = load_fixture("girr_delta_v1")
result = calculate_sbm_capital(fixture.sensitivities, context=fixture.context)
validate_sbm_result_reconciliation(result)
payload = serialize_sbm_result(result)

display(
    Markdown(
        markdown_table(
            (
                "Fixture",
                "Input rows",
                "Total capital",
                "Expected total",
                "Profile hash",
                "Input hash",
            ),
            (
                (
                    fixture.fixture_id,
                    len(fixture.sensitivities),
                    payload["total_capital"],
                    fixture.expected_outputs["total_capital"],
                    payload["profile_hash"][:16] + "...",
                    payload["input_hash"][:16] + "...",
                ),
            ),
        )
    )
)

## Implementation details / math verification - Delta audit payload

The remaining cells inspect selected scenarios, bucket records, weighted sensitivities, determinism, and validation errors.


In [ ]:
display(
    Markdown(
        markdown_table(
            ("Risk class", "Measure", "Selected scenario", "Selected capital"),
            (
                (
                    risk_class["risk_class"],
                    risk_class.get("risk_measure", ""),
                    risk_class.get("selected_scenario", ""),
                    risk_class["selected_capital"],
                )
                for risk_class in payload["risk_classes"]
            ),
        )
    )
)

In [ ]:
bucket_rows = []
for risk_class in payload["risk_classes"]:
    for bucket in risk_class["buckets"]:
        bucket_rows.append(
            (
                risk_class["risk_class"],
                bucket["bucket_id"],
                bucket["kb"],
                bucket.get("sb", ""),
                bucket.get("floor_applied", False),
            )
        )

display(
    Markdown(
        markdown_table(
            ("Risk class", "Bucket", "Kb", "Sb", "Floor applied"),
            bucket_rows,
        )
    )
)

In [ ]:
weighted_rows = []
for risk_class in payload["risk_classes"]:
    for bucket in risk_class["buckets"]:
        for item in bucket["weighted_sensitivities"]:
            weighted_rows.append(
                (
                    bucket["bucket_id"],
                    item["sensitivity_id"],
                    item["risk_weight"],
                    item["scaled_amount"],
                )
            )

display(
    Markdown(
        markdown_table(
            ("Bucket", "Sensitivity", "Risk weight", "Scaled amount"),
            weighted_rows,
        )
    )
)

In [ ]:
second_payload = serialize_sbm_result(
    calculate_sbm_capital(fixture.sensitivities, context=fixture.context)
)
display(
    Markdown(
        markdown_table(
            ("Replay check", "Result"),
            (
                ("Payload equality", second_payload == payload),
                ("Input hash equality", second_payload["input_hash"] == payload["input_hash"]),
                (
                    "Profile hash equality",
                    second_payload["profile_hash"] == payload["profile_hash"],
                ),
            ),
        )
    )
)

In [ ]:
case_id, expected_error, invalid_sensitivities = fixture.invalid_cases[0]
try:
    calculate_sbm_capital(invalid_sensitivities, context=fixture.context)
    caught = ("none", "no exception")
except (SbmInputError, UnsupportedRegulatoryFeatureError) as exc:
    caught = (type(exc).__name__, str(exc))

display(
    Markdown(
        markdown_table(
            ("Invalid case", "Expected match", "Caught type", "Caught message"),
            ((case_id, expected_error, caught[0], caught[1]),),
        )
    )
)

## See also

- [SBM package journey](../docs/PACKAGE_JOURNEY.md)
- [SBM dataset contract](../docs/DATASET_CONTRACT.md)
- [Client integration guide](../../../docs/CLIENT_INTEGRATION.md)
- [DRC package journey](../../frtb-drc/docs/PACKAGE_JOURNEY.md)
- [RRAO package journey](../../frtb-rrao/docs/PACKAGE_JOURNEY.md)
- [CVA package journey](../../frtb-cva/docs/PACKAGE_JOURNEY.md)
- [IMA package journey](../../frtb-ima/docs/PACKAGE_JOURNEY.md)
